# Mobile Game Product & Ad Monetization Analytics
- Understanding player engagement, advertising monetization, and long-term player value

### Core business question
> **How should a mobile game optimize rewarded advertising to increase monetization and long-term player value without damaging player engagement and retention?**
>
Acquisition → Engagement → Ad exposure → Monetization → Retention → LTV

#### 1. Ad Monetization Funnel & Diagnostic Analysis
#### 2. Player engagement → ad monetization → LTV
#### 3. Player segmentation & monetization personas
#### 4. Acquisition channel → retention/LTV
#### 5. Rewarded vs. other ad formats

In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/Users/asamitakeuchi/Downloads/Data/mobile-game-ltv-forecasting-challenge/train.csv')

/var/folders/1v/2zr5v4ln72s0cjq21hbhbv3w0000gn/T/ipykernel_23881/4108302586.py:1: DtypeWarning: Columns (10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/asamitakeuchi/Downloads/Data/mobile-game-ltv-forecasting-challenge/train.csv')


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21006238 entries, 0 to 21006237
Data columns (total 15 columns):
 #   Column             Dtype  
---  ------             -----  
 0   user_id            int64  
 1   platform           object 
 2   country_tier       object 
 3   channel_tier       object 
 4   install_day        int64  
 5   install_week       int64  
 6   day_since_install  int64  
 7   event_hour         int64  
 8   event_type         object 
 9   event_name         object 
 10  product_id         object 
 11  network            object 
 12  ad_placement       object 
 13  revenue_usd        float64
 14  ltv_d8_d180        float64
dtypes: float64(2), int64(5), object(8)
memory usage: 2.3+ GB


- **channel_tier**: Install media source. Channels with fewer than 50 users are grouped as other
- **install_day**: Days since the earliest install in the sample.
- **install_week**: install_day // 7
- **event_type**: Event category: session, iap (In-App Purchase), or ad_impression
- **event_name**: specific event: session_start, af_purchase / in_app_purchase, or ad format (ad_reward, ad_interstitial, etc.)
- **product_id**: Purchased item ID (af_content_id for af_purchase, af_product_id otherwise). Empty for non-IAP events.
- **network**: Ad network name. Empty for non-ad events.
- - An advertising network is a technology business that connects companies wanting to display ads (advertisers) with website or app owners who have space available to show them.
- **ad_placement**: Ad placement name. Empty for non-ad events.
- - Ad placement is the specific location on a webpage, app, or social media platform where an advertisement appears
- **revenue_usd**: Revenue in USD for IAP or ad impression events. NaN for session events.

In [7]:
df.head()

,user_id,platform,country_tier,channel_tier,install_day,install_week,day_since_install,event_hour,event_type,event_name,product_id,network,ad_placement,revenue_usd,ltv_d8_d180
0,1,android,UZ,92247aa9,6,0,0,0,ad_impression,ad_reward,NaN,54b36581,e7146d79,0.000331,6.644226
1,1,android,UZ,92247aa9,6,0,0,0,ad_impression,ad_reward,NaN,b98072a5,9e17b867,0.000226,6.644226
2,1,android,UZ,92247aa9,6,0,0,0,ad_impression,ad_reward,NaN,54b36581,e7146d79,0.000281,6.644226
3,1,android,UZ,92247aa9,6,0,0,0,ad_impression,ad_reward,NaN,54b36581,e7146d79,0.000356,6.644226
4,1,android,UZ,92247aa9,6,0,0,0,ad_impression,ad_reward,NaN,4710c070,018933a5,0.000322,6.644226


In [61]:
columns = ['user_id', 'channel_tier', 'event_type', 'event_name', 'product_id', 'network', 'ad_placement']

for col in columns: print(f'{col} number of unique values:', df[col].nunique())

user_id number of unique values: 75464
channel_tier number of unique values: 16
event_type number of unique values: 3
event_name number of unique values: 3
product_id number of unique values: 161
network number of unique values: 12
ad_placement number of unique values: 20


In [91]:
columns_vals = ['event_type', 'event_name', 'network', 'channel_tier']

for col in columns_vals: print(f'values of {col}:', df[col].unique())

values of event_type: ['ad_impression' 'session' 'iap']
values of event_name: ['ad_reward' 'session_start' 'af_purchase']
values of network: ['54b36581' 'b98072a5' '4710c070' '62865ca7' '48dbfa82' '0f95c3e4'
 '6fa365a1' '8ef7ca0d' nan 'c2b68400' 'ba78355e' '29a106dd' '72ffa408']
values of channel_tier: ['92247aa9' 'bb16a88d' '8ad56f16' '0a1068d4' '0487c9df' '0a0ae9c4' 'other'
 '3bc36bb9' 'be8c018e' '1e3b259e' '96e2a77d' 'a9603bc5' '62eb5cc1'
 'a4ec9600' '249fc9c1' 'c7b4e5c9']


In [79]:
df.describe().round(2)

,user_id,install_day,install_week,day_since_install,event_hour,revenue_usd,ltv_d8_d180
count,21006238.00,21006238.00,21006238.00,21006238.00,21006238.00,18646726.00,21006238.00
mean,37657.35,14.36,1.68,3.29,12.03,0.04,39.16
std,21143.18,9.40,1.31,2.32,6.08,1.16,423.47
min,1.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,19876.00,6.00,0.00,1.00,7.00,0.00,1.28
50%,38259.00,15.00,2.00,3.00,13.00,0.00,5.21
75%,55455.00,23.00,3.00,5.00,17.00,0.00,13.23
max,97539.00,30.00,4.00,7.00,23.00,146.27,62046.47


- 75% of events are NOT developing revenue
- Median LTV/event is 5.21

In [30]:
def null_count(df):
    results = []
    for val in df.columns:
        null_count = df[val].isnull().sum()
        ttl_count = df[val].count()
        null_pct = round(null_count/ttl_count*100.0, 2)

        results.append((val, null_count, null_pct))

    return results

In [32]:
null_count(df)

[('user_id', 0, 0.0),
 ('platform', 0, 0.0),
 ('country_tier', 0, 0.0),
 ('channel_tier', 0, 0.0),
 ('install_day', 0, 0.0),
 ('install_week', 0, 0.0),
 ('day_since_install', 0, 0.0),
 ('event_hour', 0, 0.0),
 ('event_type', 0, 0.0),
 ('event_name', 0, 0.0),
 ('product_id', 20950193, 37381.02),
 ('network', 2415572, 12.99),
 ('ad_placement', 2415572, 12.99),
 ('revenue_usd', 2359512, 12.65),
 ('ltv_d8_d180', 0, 0.0)]

In [57]:
df['product_id'].value_counts()[:20]

product_id
bcc00833    5712
e782250a    3235
4b938d57    3070
c38a022a    2837
d032a5fe    2367
7fb6e686    2309
3221acdd    2155
ba0e1e7b    2014
68a9307c    1832
9ae70675    1788
0cf337e5    1739
c0ce49e2    1657
b1dd1c41    1438
a36c6fbf    1167
e43d08a6    1104
3534c002     956
979b377d     870
983b2eeb     826
28eb399a     825
9d93913c     812
Name: count, dtype: int64

In [93]:
df.groupby(['event_type', 'event_name']).user_id.count()

event_type     event_name   
ad_impression  ad_reward        18590666
iap            af_purchase         56060
session        session_start     2359512
Name: user_id, dtype: int64

# 1. Ad Monetization Funnel & Diagnostic Analysis
### Business Question

> **Where are the biggest opportunities to improve advertising monetization across ad formats, placements, networks, platforms, and player cohorts?**
> 

The goal is to move beyond simply measuring ad impressions and identify **where monetization efficiency differs**.

### Hypothesis

**H1:** Advertising monetization efficiency varies significantly by **ad format, placement, network, platform, country, and player lifecycle stage**.

**H2:** Some placements/networks generate disproportionately high revenue efficiency despite lower impression volume.

## Funnel Analysis
### The advertising funnel at the player/event level
1. **Installed Players** = unique users who installed the game - user_id/platform, country, channel, install_week
2. **Active Players:** = users with >= 1 session_start
3. **Ad-Exposed Players** = Users with >= 1 ad impression
4. **Reward / Ad Format Exposure** = Users exposed to specific ad formats / placements
5. **Monetized Impressions** = Ad impressions generating revenue
6. **Ad Revenue** = ttl advertising revenue

### 1. Installed Players
### 2. Active Players: = users with >= 1 session_start
= unique users who installed the game - user_id/platform, country, channel, install_week

In [116]:
# Installed Players
unique_users = df['user_id'].nunique()

# Active Players = user_id with >= 1 session_start
active_users = df.loc[df['event_name'] == 'session_start', 'user_id'].nunique()

print(f'Number of Unique installed users:', unique_users)
print(f'Number of Active Players:', active_users)

Number of Unique installed users: 75464
Number of Active Players: 61800


### 3. Ad-Exposed Players = Users with >= ad impression
### 4. Reward / Ad Format Exposure = Users exposed to specific ad formats / placements

In [132]:
# Ad-exposed users
ad_exposed_users = df.loc[df['event_type'] == 'ad_impression', 'user_id'].nunique()

# Reward / Ad Format Exposure 
ad_placement_ex = df.groupby(['event_type', 'ad_placement']).user_id.count().sort_values(ascending=False)

network_ex = df.groupby(['event_type', 'network']).user_id.count().sort_values(ascending=False)

# ad_exposed_users X channel_tier OR ad_placement
print(f'Number of Ad Exposed users:', ad_exposed_users)
print(f'Exposure by Ad Placement:', ad_placement_ex)
print(f'Exposure by Network:', network_ex )

Number of Ad Exposed users: 51095
Exposure by Ad Placement: event_type     ad_placement
ad_impression  6fc118c8        3480272
               9e17b867        3332889
               02674f0c        2417876
               e7146d79        2040132
               018933a5        1659910
               6a75a998        1361893
               2401f920        1184040
               db2fa9e8        1084342
               27792d2d         652746
               05f3136b         353699
               eb5f6d53         314784
               e1e6dcbc         127175
               5c3d47b6         122017
               262e10fd         115607
               11abab9b         100180
               64c99e5c          95693
               2b2982d0          56605
               00216aa2          54540
               1e4b064c          25381
               56d1d951          10885
Name: user_id, dtype: int64
Exposure by Network: event_type     network 
ad_impression  b98072a5    3389494
               8ef7ca0d 

### 5. Monetized Impressions = Ad impressions generating revenue

In [198]:
# 5 Monetized Impressions = Ad impressions generating revenue
monetized_impressions = df.loc[df['event_type'] == 'ad_impression', 'revenue_usd'] > 0
num_ad_impression = len(df[df['event_type'] == 'ad_impression'])
ttl_ad_revenue = df.loc[df['event_type'] == 'ad_impression', 'revenue_usd'].sum()

print(f'Impressions generating revenue:', len(monetized_impressions))
print(f'Ttl ad impressions:', num_ad_impression)
print(f'Ttl ad revenue/impression:', round(ttl_ad_revenue/len(monetized_impressions), 6))
print(f'eCPM (the total ad revenue earned for every 1,000 ad impressions):', round((ttl_ad_revenue/len(monetized_impressions)*1000.00), 2))

Impressions generating revenue: 18590666
Ttl ad impressions: 18590666
Ttl ad revenue/impression: 0.004049
eCPM (the total ad revenue earned for every 1,000 ad impressions): 4.05


### 6. Ad Revenue = ttl advertising revenue

In [163]:
# 6. Ad Revenue = ttl advertising revenu
print(f'Ttl Ad revenue:', round(ttl_ad_revenue, 2))
print(f'Ad revenue per user:', round(ttl_ad_revenue/unique_users, 2))
print(f'Ad revenue per DAU:', round(ttl_ad_revenue/active_users, 2))

Ttl Ad revenue: 75272.29
Ad revenue per user: 1.0
Ad revenue per DAU: 1.22


### Guardrail metrics
- D1 retention
- D7 retention
- sessions/user
- IAP revenue/user
- total revenue/user

In [186]:
# D1 & D7 Retention
d1_retention = df.loc[df['install_day'] == 1, 'user_id'].nunique()
d7_retention = df.loc[df['install_day'] == 7, 'user_id'].nunique()

print(f'D1 retention %:', d1_retention/unique_users*100.00)
print(f'D7 retention %:', d7_retention/unique_users*100.00)

D1 retention %: 6.2864412170041355
D7 retention %: 6.173804728082264


In [194]:
# session/user, IAP revenue/user, and total revenue/user
ttl_iap_rev = df.loc[df['event_type'] == 'iap', 'revenue_usd'].sum()
ttl_revenue = df['revenue_usd'].sum()

print(f'Ad revenue vs IAP revenue - Ad rev: {round(ttl_ad_revenue, 2)}, IAP rev {round(ttl_iap_rev, 2)}')
print(f'IAP revenue/user:', round(ttl_iap_rev/unique_users, 2))
print(f'Ttl revenue/user:', round(ttl_revenue/unique_users, 2))

Ad revenue vs IAP revenue - Ad rev: 75272.29, IAP rev 713261.25
IAP revenue/user: 9.45
Ttl revenue/user: 10.45


### Result of the Funnel Analysis

## 2. Network × Placement performance matrix

In [ ]:
# Develop eCPM and Revenue/User column in df
df['eCPM']

In [205]:
df.groupby(['network', 'ad_placement']).user_id.count().sort_values(ascending=False)

network   ad_placement
b98072a5  9e17b867        3332889
8ef7ca0d  6fc118c8        2826755
c2b68400  02674f0c        2417876
54b36581  e7146d79        2040132
4710c070  018933a5        1659910
ba78355e  6a75a998        1361893
29a106dd  2401f920        1184040
0f95c3e4  db2fa9e8        1084342
62865ca7  6fc118c8         653517
48dbfa82  27792d2d         652746
6fa365a1  05f3136b         353699
72ffa408  eb5f6d53         314784
4710c070  e1e6dcbc         127175
0f95c3e4  5c3d47b6         122017
54b36581  262e10fd         115607
6fa365a1  11abab9b         100180
ba78355e  64c99e5c          95693
b98072a5  2b2982d0          56605
48dbfa82  00216aa2          54540
29a106dd  1e4b064c          25381
c2b68400  56d1d951          10885
Name: user_id, dtype: int64

In [207]:
df.groupby(['network', 'ad_placement', 'event_type']).user_id.count().sort_values(ascending=False)

network   ad_placement  event_type   
b98072a5  9e17b867      ad_impression    3332889
8ef7ca0d  6fc118c8      ad_impression    2826755
c2b68400  02674f0c      ad_impression    2417876
54b36581  e7146d79      ad_impression    2040132
4710c070  018933a5      ad_impression    1659910
ba78355e  6a75a998      ad_impression    1361893
29a106dd  2401f920      ad_impression    1184040
0f95c3e4  db2fa9e8      ad_impression    1084342
62865ca7  6fc118c8      ad_impression     653517
48dbfa82  27792d2d      ad_impression     652746
6fa365a1  05f3136b      ad_impression     353699
72ffa408  eb5f6d53      ad_impression     314784
4710c070  e1e6dcbc      ad_impression     127175
0f95c3e4  5c3d47b6      ad_impression     122017
54b36581  262e10fd      ad_impression     115607
6fa365a1  11abab9b      ad_impression     100180
ba78355e  64c99e5c      ad_impression      95693
b98072a5  2b2982d0      ad_impression      56605
48dbfa82  00216aa2      ad_impression      54540
29a106dd  1e4b064c      ad_impr